# Milestone 5 — VLM Membership Inference Attack

**New in Milestone 5:**
- Min-K%++ features (z-score normalized)
- Zlib compression ratio baseline
- Temperature sensitivity features (Hu et al. 2025)
- Additional Min-K% thresholds (k=5%, k=30%)
- Random Forest + LightGBM classifiers
- Feature engineering (squared + interaction terms)
- Weighted ensemble
- Formatted results table with standalone baselines

In [ ]:
import sys
import os

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/milestone5/'

sys.path.insert(0, SAVE_DIR)

print(f'sys.path after insert: {sys.path}')
if os.path.exists(SAVE_DIR):
    print(f'Contents of {SAVE_DIR}: {os.listdir(SAVE_DIR)}')
else:
    print(f'{SAVE_DIR} does not exist — creating it.')
    os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
!pip install -q xgboost lightgbm
!pip install -q num2words

In [ ]:
import os
import pickle
import warnings
warnings.filterwarnings('ignore')
import torch

from models import load_data, load_model_and_processor, load_base_model_and_processor
from batches import process_dataset_in_batches
from classifier import build_membership_classifier, generate_submission

os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Always load splits — needed for labels and test IDs even when using cached features.
splits = load_data()

## 1. Extract Features

Run the cells in this section to extract features from scratch.
Skip to **Section 2** if you already have cached CSVs on Drive.

> **Note:** Features must be re-extracted with the new `features.py` to include
> temperature sensitivity (`loss_at_T05`, `loss_at_T20`, `temp_sensitivity`),
> Min-K%++ (`min_k10_pp`, `min_k20_pp`), and zlib ratio. Old cached CSVs from
> Milestone 4 will still work with the classifier (missing columns are skipped),
> but you will miss the new features.

In [ ]:
# Load fine-tuned (target) and base (reference) models
processor, model = load_model_and_processor(device)
base_processor, base_model = load_base_model_and_processor(device)

In [ ]:
train_features = process_dataset_in_batches(
    model, processor, splits['train'], has_labels=True, batch_size=200,
    base_model=base_model, base_processor=base_processor
)
train_features.to_csv(f'{SAVE_DIR}train.csv', index=False)
print('Train features saved.')

In [ ]:
val_features = process_dataset_in_batches(
    model, processor, splits['validation'], has_labels=True, batch_size=200,
    base_model=base_model, base_processor=base_processor
)
val_features.to_csv(f'{SAVE_DIR}val.csv', index=False)
print('Val features saved.')

In [ ]:
test_features = process_dataset_in_batches(
    model, processor, splits['test'], has_labels=False, batch_size=200,
    base_model=base_model, base_processor=base_processor
)
test_features.to_csv(f'{SAVE_DIR}test.csv', index=False)
print('Test features saved.')

## 2. Load Cached Features

Run this cell instead of Section 1 when resuming a session.

In [ ]:
import pandas as pd
train_features = pd.read_csv(f'{SAVE_DIR}train.csv')
val_features   = pd.read_csv(f'{SAVE_DIR}val.csv')
test_features  = pd.read_csv(f'{SAVE_DIR}test.csv')
print('Features loaded.')
print(f'  train: {train_features.shape}  val: {val_features.shape}  test: {test_features.shape}')
print(f'  Columns: {list(train_features.columns)}')

## 3. Train Classifiers & Evaluate

Trains standalone baselines (Loss, Zlib Ratio, Min-K%, Min-K%++) plus
Logistic Regression, Random Forest, XGBoost, LightGBM, and a weighted ensemble.
Results are printed as a formatted table.

In [ ]:
results = build_membership_classifier(train_features, val_features)

best_name = max(
    (n for n in results if n not in {"Loss", "Zlib Ratio", "Min-K% (k=10%)", "Min-K%++ (k=10%)"}),
    key=lambda x: results[x]['auc']
)
best = results[best_name]
print(f'\nSelected for submission: {best_name}  AUC={best["auc"]:.4f}')

## 4. Generate Submission

In [ ]:
classifier_path = f'{SAVE_DIR}classifier.pkl'
with open(classifier_path, 'wb') as f:
    pickle.dump(best, f)
print(f'Classifier saved to {classifier_path}')

submission = generate_submission(
    splits['test'], test_features, best,
    output_path='submission.csv'
)
print(submission.head())

In [ ]:
from google.colab import files
files.download('submission.csv')